- **학습 목표**: `sigmoid` 함수를 수치적으로 안전하게 직접 구현하고, 왜/어디서 오버플로가 나는지 스스로 확인한다.
- **핵심 개념**: 표준 시그모이드는 $1/(1+e^{-z})$. `np.exp`는 float64 기준 인자가 약 709.78($=\ln(\text{float64 max})$)을 넘으면 오버플로 난다. 시그모이드는 이 식 때문에 $z$가 **매우 큰 음수**일 때 위험해진다($-z$가 커지므로) — 반대로 $z$가 큰 양수일 때는 위험하지 않다는 비대칭성이 있다.

In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss

In [2]:
def sigmoid(z: np.ndarray) -> np.ndarray:
    """
    요구사항: 수치적 오버플로를 방지할 것 (z가 매우 크거나 작을 때 exp가 터짐).
    어떻게 방지할지는 스스로 결정 (힌트: np.clip 또는 조건 분기).
    """
    z = np.clip(z, -500, 500)
    result = 1 / (1 + np.exp(-z))
    return result

# 감각 확인: overflow 임계값 근방에서 정상적으로 동작하는지
print(sigmoid(np.array([-1000.0, -2.0, 0.0, 2.0, 1000.0])))

[7.12457641e-218 1.19202922e-001 5.00000000e-001 8.80797078e-001
 1.00000000e+000]


**확인 질문:** float64에서 `np.exp(x)`는 몇 부터 오버플로가 나는가? 시그모이드 식 $1/(1+e^{-z})$ 기준으로, `z`가 매우 큰 **음수**일 때와 매우 큰 **양수**일 때 중 어느 쪽이 위험한가?

**답변:** `np.exp(x)`는 $x \gtrsim 709.78$부터 오버플로($\because$ float64 최댓값 $\approx 1.8\times10^{308}$, $\ln(1.8\times10^{308})\approx 709.78$). 시그모이드는 $e^{-z}$를 계산하므로 `z`가 매우 큰 **음수**일 때 `-z`가 커져서 위험하다. `z`가 매우 큰 양수일 때는 `e^{-z}`가 0에 가까워질 뿐이라 안전하다. `np.clip(z, -500, 500)`으로 이 비대칭 문제를 안전 마진을 두고 방지.

---
- **학습 목표**: 로지스틱 회귀를 sklearn 없이 배치 경사하강으로 직접 학습시킨다 (Day3 오전에 유도한 gradient 사용).
- **핵심 개념**: 매 iteration마다 `z = Xw+b` → `p = sigmoid(z)` → NLL(cross-entropy) 손실 계산·기록 → gradient로 `w, b` 업데이트. 손실은 $-\frac{1}{n}\sum[y\log p + (1-y)\log(1-p)]$이고, gradient는 $\frac{\partial L}{\partial w}=\frac{1}{n}X^T(p-y)$, $\frac{\partial L}{\partial b}=\frac{1}{n}\sum(p-y)$.

In [3]:
def train_logistic_regression(X, y, lr=0.1, n_iters=2000):
    """
    요구사항:
    - w를 0으로, b를 0.0으로 초기화.
    - 매 반복마다: z 계산 → p = sigmoid(z) → 음의 로그우도 손실 계산 → 손실 기록
      → Day3 오전에 유도한 gradient로 w, b 업데이트.
    - log(0) 방지책을 넣을 것 (어떻게 할지 스스로 결정).
    - (w, b, losses) 반환.
    """
    n_samples, n_features = X.shape

    w = np.zeros(n_features)
    b = 0.0

    losses = []

    for i in range(n_iters):
        z = X @ w + b
        p = sigmoid(z)
        eps = 1e-15
        p = np.clip(p, eps, 1 - eps)
        loss = -1 / n_samples * np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))

        losses.append(loss)

        dw = 1 / n_samples * X.T @ (p - y)
        db = np.mean(p - y)

        w -= lr * dw
        b -= lr * db

    return w, b, losses

**디버깅 기록 (요약):** 처음 구현에서 세 번의 버그를 스스로 잡아냈다.
1. `w`를 파이썬 리스트로 초기화 + `dw = X@(y-p)`가 `X.T`가 아니라 `X`를 써서 shape mismatch, `db = y-p`가 스칼라로 축약 안 됨 → 셋 다 수정.
2. `dw = -1/n * X.T@(p-y)`처럼 마이너스가 다시 붙어 부호가 반전되고, `db`는 `np.mean`과 `1/n`을 이중으로 곱해 `1/n²` 스케일이 됨 → 15 iteration만 돌려도 손실이 `0.693→1.038`로 증가하는 걸로 확인, 부호/스케일 수정.
3. `sigmoid`에서 `np.exp(z)`로 써서(음수 부호 누락) 실제로는 `sigmoid(-z)`를 계산 → `make_classification` 데이터에서 손실이 `0.693→31.58`로 폭증하는 것으로 발견, `sigmoid`만 수정.

세 버그를 모두 고친 뒤 합성 데이터(참값 $w=[1.5,-2.0,1.0]$)로 검증: 손실이 `0.693→0.089`로 단조 감소, 학습된 `w`가 참값과 부호·상대크기 패턴 일치.

In [4]:
# 합성 데이터로 최종 검증 (참값 w와 부호·크기 패턴이 맞는지 확인)
rng = np.random.default_rng(0)
n_samples, n_features = 200, 3
X_syn = rng.standard_normal((n_samples, n_features))
true_w = np.array([1.5, -2.0, 1.0])
y_syn = (sigmoid(X_syn @ true_w) > 0.5).astype(float)

w_syn, b_syn, losses_syn = train_logistic_regression(X_syn, y_syn)
print("loss[0], loss[-1]:", losses_syn[0], losses_syn[-1])
print("학습된 w:", w_syn, " (참값 비율 [1.5, -2.0, 1.0])")
print("단조 감소 여부:", all(losses_syn[i+1] <= losses_syn[i] + 1e-9 for i in range(len(losses_syn)-1)))

loss[0], loss[-1]: 0.6931471805599452 0.1000304252333385
학습된 w: [ 3.82626207 -5.09178696  2.73190069]  (참값 비율 [1.5, -2.0, 1.0])
단조 감소 여부: True


---
- **학습 목표**: 직접 구현한 로지스틱 회귀를 `sklearn.LogisticRegression`과 테스트 정확도·log_loss로 비교해 구현이 맞는지 검증한다.
- **핵심 개념**: 정확도만 보면 데이터가 너무 쉬울 때(작은 샘플 수, 잘 분리된 클래스) 두 구현이 똑같이 만점이 나와서 비교가 무의미해질 수 있다. 또한 sklearn의 `LogisticRegression`은 기본으로 L2 정규화가 켜져 있어(`C=1.0`), 정규화 없는 내 구현과 손실을 비교할 때 완전히 같은 값이 나오길 기대하면 안 된다.

In [5]:
def predict_labels(X, w, b, threshold=0.5):
    p = sigmoid(X @ w + b)
    return np.array([1 if pi >= threshold else 0 for pi in p])

def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

In [6]:
# 일부러 어려운(클래스가 잘 안 갈리는) 데이터로 비교 — 너무 쉬우면 두 구현 다 정확도 1.0이 나와 비교가 무의미해짐
X, y = make_classification(
    n_samples=500, n_features=5, n_informative=3,
    class_sep=0.5, flip_y=0.1, random_state=42,
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 1) 내 구현
w, b, losses = train_logistic_regression(X_train, y_train)
y_pred_mine = predict_labels(X_test, w, b)
acc_mine = accuracy(y_test, y_pred_mine)
loss_mine = log_loss(y_test, sigmoid(X_test @ w + b))

# 2) sklearn (기본값, L2 정규화 켜짐)
sk_l2 = LogisticRegression(max_iter=1000)
sk_l2.fit(X_train, y_train)
acc_sk_l2 = sk_l2.score(X_test, y_test)
loss_sk_l2 = log_loss(y_test, sk_l2.predict_proba(X_test)[:, 1])

# 3) sklearn (정규화 끔 — 내 구현과 같은 목적함수로 공정 비교)
sk_none = LogisticRegression(max_iter=1000, penalty=None)
sk_none.fit(X_train, y_train)
acc_sk_none = sk_none.score(X_test, y_test)
loss_sk_none = log_loss(y_test, sk_none.predict_proba(X_test)[:, 1])

print(f"정확도   - 내 구현: {acc_mine:.4f}  sklearn(L2): {acc_sk_l2:.4f}  sklearn(정규화 없음): {acc_sk_none:.4f}")
print(f"log_loss - 내 구현: {loss_mine:.6f}  sklearn(L2): {loss_sk_l2:.6f}  sklearn(정규화 없음): {loss_sk_none:.6f}")

정확도   - 내 구현: 0.7333  sklearn(L2): 0.7333  sklearn(정규화 없음): 0.7333
log_loss - 내 구현: 0.619614  sklearn(L2): 0.618002  sklearn(정규화 없음): 0.619596


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


**결론:** 정확도는 세 경우 모두 동일하게 나오고, `log_loss`는 정규화가 켜진 sklearn과는 약간 차이가 나지만(정규화가 확신을 눌러주는 효과), 정규화를 끈 sklearn과는 소수점 여러 자리까지 거의 일치한다. 즉 내가 만든 `sigmoid` + `train_logistic_regression`이 sklearn의 (정규화 없는) 로지스틱 회귀와 **동일한 목적함수를 실제로 올바르게 최소화하고 있다**는 것을 확인했다.

**다음에 볼 것:** 로지스틱 회귀에서 MSE 대신 cross-entropy를 쓰는 이유 (키워드: sigmoid+MSE 결합 시 손실함수의 non-convexity, gradient vanishing 구간).